# EXP-017 — Lựa chọn đặc trưng KNN trên WESAD

**Người thực hiện:** Nam  
**Nhóm:** 04 — Feature Selection  
**Trạng thái:** Chưa chạy trong bản mã nguồn phát hành.

## 1. Mục tiêu và giả thuyết

So sánh KNN mặc định dùng đủ 68 đặc trưng và KNN chỉ dùng 30 đặc trưng theo SelectKBest (ANOVA F-test). Giả thuyết: giảm chiều có thể loại bỏ các đặc trưng ít hữu ích và cải thiện tổng quát hóa.

## 2. Thiết kế thí nghiệm

- G2 physiological68: 15 người, cùng bảng với EXP-003.
- LOSO 15 người, hai bài toán nhị phân/ba lớp.
- Hai biến thể trên cùng outer-fold: `knn_68` và `knn_top30`.
- Mỗi fold: imputer → scaler → SelectKBest → KNN; tất cả fit **chỉ trên train**, tuyệt đối không fit selector trên toàn bảng.
- KNN mặc định k=5, Euclidean, uniform; không tối ưu tham số. Tổng 60 model fits.

## 3. Khởi tạo môi trường và đọc cấu hình

In [ ]:
from pathlib import Path
import sys, json, hashlib, subprocess
import pandas as pd
import yaml
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'scripts/run_experiment.py').is_file() and (p/'src/wesad/models.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Không tìm thấy wesad-stress-detection. Hãy mở notebook trong repo đúng.')
print('ROOT:', ROOT)
print('Python:', sys.executable)

def sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

def execute(args):
    command=[sys.executable,*args]
    print('Lệnh:', ' '.join(map(str,command)))
    proc=subprocess.run(command,cwd=ROOT,capture_output=True,text=True,encoding='utf-8',errors='replace')
    print('Return code:',proc.returncode)
    if proc.stdout: print('STDOUT: \n'+proc.stdout[-12000:])
    if proc.stderr: print('STDERR: \n'+proc.stderr[-12000:])
    if proc.returncode: raise RuntimeError(f'Chạy thất bại ({proc.returncode}). Xem STDERR ở trên.')
    return proc

CONFIG_PATH=ROOT/'experiments/04_improvements/EXP-017_KNN_Feature_Selection/config.yaml'
EXPORT_DIR=CONFIG_PATH.parent
assert CONFIG_PATH.is_file()
print(CONFIG_PATH)

## 4. Kiểm tra dữ liệu và checksum

In [ ]:
CONFIG=yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
DATA=ROOT/CONFIG['features']; SCHEMA=ROOT/CONFIG['schema']
for f in [DATA,SCHEMA]:
    if not f.is_file(): raise FileNotFoundError(f'Thiếu dữ liệu G2: {f}')
actual=sha256(DATA)
assert actual==CONFIG['feature_sha256'],f'Không khớp checksum: {actual}'
sys.path.insert(0,str(ROOT/'src'))
from wesad.data import load_feature_table
frame, features=load_feature_table(DATA,SCHEMA)
assert len(features)==68 and frame.subject_id.nunique()==15
print('Số cửa sổ:',len(frame),'| Số người:',frame.subject_id.nunique(),'| Số đặc trưng:',len(features))
display(frame.label_original.value_counts().sort_index().rename('Số cửa sổ').to_frame())


## 5. Kiểm tra cấu hình KNN và bộ chọn đặc trưng

In [ ]:
sys.path.insert(0,str(ROOT/'scripts'))
from run_exp017 import make_pipeline
assert CONFIG['selected_features']==30
print('Đối chứng KNN 68:',make_pipeline())
print('Cải tiến KNN 30:',make_pipeline(CONFIG['selected_features']))


## 6. R01 — Chạy thử S2

Bật `RUN_QUICK=True` để chạy 2 bài toán × 2 biến thể trên S2. Kết quả quick được tách riêng ở `run_outputs/quick_S2/`.

In [ ]:
RUN_QUICK=False
if RUN_QUICK:
    execute(['scripts/run_exp017.py','--config',str(CONFIG_PATH),'--quick-subject','S2'])
else: print('Chưa chạy quick. Bật RUN_QUICK=True nếu cần.')


## 7. R02 — Chạy LOSO chính thức đủ 15 người

Bật `RUN_MAIN=True`. Script huấn luyện rồi tự xuất CSV và PNG **ngay trong EXP-017_KNN_Feature_Selection**, không phải thêm bước copy từ outputs.

In [ ]:
RUN_MAIN=False
if RUN_MAIN:
    execute(['scripts/run_exp017.py','--config',str(CONFIG_PATH)])
else: print('Chưa chạy main. Đặt RUN_MAIN=True để chạy 60 fits.')


## 8. Đọc CSV đã tạo

In [ ]:
summary_path=EXPORT_DIR/'summary.csv'
if summary_path.is_file():
    display(pd.read_csv(summary_path))
    display(pd.read_csv(EXPORT_DIR/'comparison_delta.csv'))
    frequency_file=EXPORT_DIR/'feature_selection_frequency.csv'
    if frequency_file.is_file(): display(pd.read_csv(frequency_file).head(15))
else: print('Chưa có dữ liệu thực nghiệm. Hãy chạy R02.')


## 9. Biểu đồ và đối chiếu EXP-003

In [ ]:
figures=sorted((EXPORT_DIR/'figures').glob('*.png'))
print('Đã có',len(figures),'ảnh PNG:')
for f in figures: print('-', f.name)
print('comparison_delta.csv là top-30 trừ KNN-68 được chạy cùng fold trong EXP-017.')
print('Chỉ ghi là so với EXP-003 khi xác minh cùng dữ liệu/phương pháp; không gán nhãn nhầm kết quả nội bộ.')


## 10. Hạn chế

ANOVA F-test đánh giá từng đặc trưng riêng lẻ và không mô hình hóa tương tác giữa các đặc trưng. Chỉ khảo sát top-30; không tự tuyên bố đây là ngưỡng tối ưu. Chọn đặc trưng trong mỗi fold tránh rò rỉ dữ liệu, nhưng không loại bỏ mọi biến thiên giữa người.

## 11. Nhận xét và kết luận

**Điền khi đã có số liệu thật:**

- KNN-68 vs KNN-top30 theo Macro-F1 (mean ± std) và Balanced Accuracy ở hai bài toán.
- Chênh lệch trung bình và fold tăng/giảm.
- Top đặc trưng xuất hiện nhiều lần từ `feature_selection_frequency.csv`.
- Đánh giá xem giảm 68 → 30 có làm tăng hiệu quả hay chỉ giảm kích thước mô hình. Không dùng số giả định.